### CS 180 Machine Exercise 1 (ME1): SMS Spam Classification with Naive Bayes
By: Alessandro Crisostomo

**Objective**

Build and hypertune a Naive Bayes classifier to detect spam messages using the SMS Spam Collection dataset. The model should achieve ≥96 % accuracy on the test set.

**Dataset**
- Use the SMS Spam Collection Dataset (available on UCI Machine Learning Repository (archive.ics.uci.edu).
- It contains 5,574 SMS messages labeled as ham (legitimate) or spam.


**Steps**

**1. Load and Explore the Dataset** (10 points)
- Import the dataset into a pandas DataFrame.
- Inspect class distribution (ham vs spam).
- Perform basic text preprocessing (lowercasing, removing punctuation, optional stopword removal).


In [11]:
#Place code here
import pandas as pd
import string

df = pd.read_csv('./sms+spam+collection/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

# Class Distribution Inspection
print(df['label'].value_counts())

# Preprocess function
def preprocess(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['clean_message'] = df['message'].apply(preprocess)
print(df.head())


label
ham     4825
spam     747
Name: count, dtype: int64
  label                                            message  \
0   ham  Go until jurong point, crazy.. Available only ...   
1   ham                      Ok lar... Joking wif u oni...   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...   
3   ham  U dun say so early hor... U c already then say...   
4   ham  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in 2 a wkly comp to win fa cup fina...  
3        u dun say so early hor u c already then say  
4  nah i dont think he goes to usf he lives aroun...  


**2. Feature Extraction** (20 points)
- Use scikit-learn’s CountVectorizer or TfidfVectorizer to convert text into numerical features.
- Experiment with:
- ngram_range (e.g., (1,1), (1,2))
- max_features (e.g., 5000, 10000)
- stop_words (e.g., "english")


In [35]:
# Place code here
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


# Count Vectorizer
vectorizer_stop = CountVectorizer(stop_words='english', ngram_range=(1, 2), max_features=10000)
X_stop = vectorizer_stop.fit_transform(df['clean_message'])

print("Features without stop words:", vectorizer_stop.get_feature_names_out())

# TFIDF-Vectorizer
# Use the counts from our previous CountVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=10000)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_message'])

print("TF-IDF Feature Names:", tfidf_vectorizer.get_feature_names_out())

Features without stop words: ['008704050406' '008704050406 sp' '01223585334' ... 'zed 08701417012150p'
 'zoe' 'üll']
TF-IDF Feature Names: ['008704050406' '008704050406 sp' '01223585334' ... 'zed 08701417012150p'
 'zoe' 'üll']


**3. Train-Test Split** (5 points)
- Split dataset into 80% training, 20% testing using train_test_split.


In [36]:
# Put code here
from sklearn.model_selection import train_test_split

X = df['clean_message']
y = df['label']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=180
)

print("Training Samples: ", len(X_train))
print("Testing Samples: ", len(X_test))

Training Samples:  4457
Testing Samples:  1115


**4. Model Training** (40 points)

- Use Multinomial Naive Bayes (MultinomialNB) from scikit-learn.
- Hypertune parameters:
- alpha (smoothing parameter, try values like 0.1, 0.5, 1.0)
- Feature extraction parameters (from step 2).


In [37]:
# Put code here
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

# Build a pipeline: TF-IDF Vectorizer -> Multinomial Naive Bayes
pipeline_count = make_pipeline(vectorizer_stop, MultinomialNB())
pipeline_tfidf = make_pipeline(tfidf_vectorizer, MultinomialNB())

# Define the parameters to test

count_param_grid = {
    'countvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3), (1, 4)],
    'multinomialnb__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
}
# 'multinomialnb__alpha' follows the naming convention: [lowercase_model_name]__[parameter]
tfidf_param_grid = {
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3), (1, 4)], # Test single words vs word pairs
    'multinomialnb__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
}

# Set up the GridSearch
grid_count = GridSearchCV(pipeline_count, count_param_grid, cv=5)
grid_tfidf = GridSearchCV(pipeline_tfidf, tfidf_param_grid, cv=5)
grid_count.fit(X_train, y_train)
grid_tfidf.fit(X_train, y_train)

print(f"Count Best parameters: {grid_count.best_params_}")
print(f"TFIDF Best parameters: {grid_tfidf.best_params_}")
print(f"Count Best cross-validation score: {grid_count.best_score_:.3f}")
print(f"TFIDF Best cross-validation score: {grid_tfidf.best_score_:.3f}")


Count Best parameters: {'countvectorizer__ngram_range': (1, 2), 'multinomialnb__alpha': 0.5}
TFIDF Best parameters: {'multinomialnb__alpha': 0.1, 'tfidfvectorizer__ngram_range': (1, 2)}
Count Best cross-validation score: 0.983
TFIDF Best cross-validation score: 0.981


**5. Evaluation Metrics** (20 points)
- Measure accuracy, precision, recall, and F1-score.
- Ensure accuracy ≥95% on the test set.


In [40]:
# Put code here
# Use the best estimator found by GridSearch
count_best_model = grid_count.best_estimator_
tfidf_best_model = grid_tfidf.best_estimator_

count_final_labels = count_best_model.predict(X_test)
tfidf_final_labels = tfidf_best_model.predict(X_test)

target_names = sorted(df['label'].unique())

# Print a quick classification report
from sklearn.metrics import classification_report, accuracy_score
count_accuracy = accuracy_score(y_test, count_final_labels)
print("COUNT Report:\n", classification_report(y_test, count_final_labels, target_names=target_names))
print("COUNT Accuracy: ", count_accuracy)

tfidf_accuracy = accuracy_score(y_test, tfidf_final_labels)
print("TFIDF Report:\n", classification_report(y_test, tfidf_final_labels, target_names=target_names))
print("TFIDF Accuracy: ", tfidf_accuracy)

COUNT Report:
               precision    recall  f1-score   support

         ham       0.99      0.99      0.99       979
        spam       0.96      0.90      0.93       136

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

COUNT Accuracy:  0.9838565022421525
TFIDF Report:
               precision    recall  f1-score   support

         ham       0.98      1.00      0.99       979
        spam       1.00      0.88      0.93       136

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.99      0.98      0.98      1115

TFIDF Accuracy:  0.9847533632286996


**6. Reflection & Insights** (5 points)
- Explains which preprocessing and parameter choices improved performance


Explanation:

Firstly, in the preprocessing stage, I lowercased all the words and letters to ensure uniformity. Furthermore, I also removed the punctuations in the sentences along with the english stop words. Then, I tried both the CountVectorizer and the TFIDFVectorizer to compare which ones would perform better. I tried different ngram ranges and (1, 2) gave the best result. I also tried different alpha smoothing parameters such as 0.01, 0.1, 0.5, 1.0, 1.5, and 2.0. For the Count Vectorizer, alpha=0.5 worked the best while alpha=0.1 was the best for the TFIDF Vectorizer. The max_features parameter also helped the performance improve from 5000 features to 10000 features. In the end, using the CountVectorizer yielded a 98.39% accuracy while TFIDFVectorizer yielded a 98.48% accuracy.